# Exploratory Corpus Characterization

**Forensic question:** What does the corpus look like before any analysis?

**Input artifacts:**
- `data/articles.db`

**Output artifacts:**
- (figures only)

**Run metadata:** (auto-populated by first code cell)


In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_settings

settings = get_settings()
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

config_hash = compute_config_hash(settings)
corpus_hash = compute_corpus_hash(settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Python | `{sys.version}` |
""")
)

In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+png'

import plotly.express as px
import polars as pl

from forensics.config import get_settings

settings = get_settings()
from forensics.utils.charts import register_forensics_template

register_forensics_template()
uri = f"sqlite:///{settings.db_path}"
q = "SELECT word_count FROM articles WHERE word_count > 0"
try:
    wc = pl.read_database_uri(q, uri)["word_count"].to_list()
    fig = px.histogram(pl.DataFrame({"word_count": wc}), x="word_count", nbins=60, title="Article word counts")
    fig.show()
except Exception as exc:
    print("Histogram skipped:", exc)

**Summary finding:** Baseline distributions and cadence views contextualize later feature shifts and change-point evidence.
